# Stylepal RAG Advanced Retrieval Evaluation

Evaluate RAG with advanced retrieval techniques (from AIE9/11_Advanced_Retrieval):

1. **Baseline** – Original `rag.retrieve()` (vector similarity, top-k)
2. **Reranking** – Retrieve more (15), Cohere rerank to top 5
3. **Multi-query** – LLM expands query into variants, union of retrievals
4. **Both** – Multi-query first, then rerank

Ragas metrics: Faithfulness, Answer Relevancy, Context Precision, Context Recall, etc.

## Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "eval":
    project_root = project_root.parent
backend_dir = project_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from dotenv import load_dotenv
load_dotenv(project_root / ".env", override=True)
_ = load_dotenv(backend_dir / ".env", override=True)

## Shared: RAG Pipeline & Evaluator

In [14]:
import os
import pandas as pd
from pathlib import Path
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, ContextPrecision, NoiseSensitivity

from services import rag, rag_advanced

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0, google_api_key=os.getenv("GEMINI_API_KEY"))
QA_PROMPT = ChatPromptTemplate.from_template(
    """Answer the question based only on the context below. Be concise.

Context:
{context}

Question: {question}

Answer:"""
)

def run_rag_pipeline(question: str, mode: str = "baseline", model: str = "gemini-2.0-flash") -> tuple[str, list[str]]:
    """Retrieve + generate. mode: baseline, rerank, multi_query, both."""
    if model == "openai":
        from langchain_openai import ChatOpenAI
        _root = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
        env_file = _root / ".env"
        api_key = None
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                if line.strip().startswith("OPENAI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found in project root .env")
        _llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=api_key)
    elif model == "gemini-2.0-flash":
        _llm = llm
    else:
        _llm = ChatGoogleGenerativeAI(model=model, temperature=0, google_api_key=os.getenv("GEMINI_API_KEY"))

    if mode == "baseline":
        docs = rag.retrieve(question, top_k=5)
        contexts = [d["content"] for d in docs]
    else:
        hits = rag_advanced.retrieve_advanced(query=question, mode=mode, top_k=5, llm=_llm)
        contexts = [h["content"] for h in hits]

    context_str = "\n\n".join(contexts) if contexts else "(No context retrieved)"
    chain = QA_PROMPT | _llm | StrOutputParser()
    answer = chain.invoke({"context": context_str, "question": question})
    return answer, contexts

# Evaluator: GPT 4.1 (avoids Gemini rate limits)
_root = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
env_file = _root / ".env"
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key and env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.strip().startswith("OPENAI_API_KEY="):
            openai_api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
            break
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=openai_api_key))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small", api_key=openai_api_key))

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_47461/3119372941.py:11: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, ContextPrecision, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_47461/3119372941.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, ContextPrecision, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_47461/3119372941.py:11: Dep

## Eval Dataset

In [12]:
EVAL_SAMPLES = [
    {
        "user_input": "What do horizontal lines do to the body?",
        "reference_contexts": ["Horizontal lines move the eye across the body, so they appear to add width and make a person appear heavier and shorter."],
        "reference": "Horizontal lines draw the eye across the body and appear to add width, making a person look heavier and shorter.",
    },
    {
        "user_input": "What is the hourglass silhouette?",
        "reference_contexts": ["Hourglass: Feminine silhouette emphasizes bust and hips and highlights a small waist."],
        "reference": "The hourglass silhouette emphasizes bust and hips with a small waist, creating a feminine shape.",
    },
    {
        "user_input": "How can color be used to minimize size?",
        "reference_contexts": ["When light and dark colors are worn together, dark clothes minimize, while light clothes emphasize."],
        "reference": "Dark colors minimize size when worn with light colors; light colors emphasize.",
    },
    {
        "user_input": "What do vertical lines do for appearance?",
        "reference_contexts": ["Vertical lines lead the eye up and down so they add height to the body and make it appear narrower."],
        "reference": "Vertical lines lead the eye up and down, adding height and making the body appear narrower.",
    },
    {
        "user_input": "What is the golden mean in fashion?",
        "reference_contexts": ["Golden Mean Belief that 3:5 proportions and 5:8 proportions are pleasing to the eye."],
        "reference": "The golden mean is the belief that 3:5 and 5:8 proportions are pleasing to the eye.",
    },
]

## Run RAG Pipeline for Each Mode

In [6]:
MODES = ["baseline", "rerank", "multi_query", "both"]
all_results = {}

for mode in MODES:
    print(f"Running {mode}...")
    results = []
    for sample in EVAL_SAMPLES:
        answer, contexts = run_rag_pipeline(sample["user_input"], mode=mode)
        results.append({
            "user_input": sample["user_input"],
            "reference_contexts": sample["reference_contexts"],
            "reference": sample["reference"],
            "response": answer,
            "retrieved_contexts": contexts,
        })
    all_results[mode] = pd.DataFrame(results)
    print(f"{mode}: {len(results)} samples")

all_results["baseline"]

Running baseline...
baseline: 5 samples
Running rerank...
rerank: 5 samples
Running multi_query...
multi_query: 5 samples
Running both...
both: 5 samples


,user_input,reference_contexts,reference,response,retrieved_contexts
0,What do horizontal lines do to the body?,[Horizontal lines move the eye across the body...,Horizontal lines draw the eye across the body ...,"Horizontal lines move the eye across the body,...",[Line \n \n• Horizontal lines move the eye acr...
1,What is the hourglass silhouette?,[Hourglass: Feminine silhouette emphasizes bus...,The hourglass silhouette emphasizes bust and h...,A silhouette that emphasizes the bust and hips...,[Hourglass\nYou are an Hourglass if..\n..your ...
2,How can color be used to minimize size?,"[When light and dark colors are worn together,...",Dark colors minimize size when worn with light...,"Dark clothes minimize size. Additionally, mono...",[• How to create pleasing proportions and avoi...
3,What do vertical lines do for appearance?,[Vertical lines lead the eye up and down so th...,"Vertical lines lead the eye up and down, addin...",They add height to the body and make it appear...,[Line \n \n• Horizontal lines move the eye acr...
4,What is the golden mean in fashion?,[Golden Mean Belief that 3:5 proportions and 5...,The golden mean is the belief that 3:5 and 5:8...,The belief that 3:5 and 5:8 proportions are pl...,[• Crew neck High plain neckline with knit ri...


## Evaluate Each Mode with Ragas

In [15]:
RAGAS_METRICS = [
    LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(),
    ContextEntityRecall(), ContextPrecision(), NoiseSensitivity()
]

eval_results = {}
for mode in MODES:
    print(f"Evaluating {mode}...")
    eval_dataset = EvaluationDataset.from_pandas(all_results[mode])
    result = evaluate(
        dataset=eval_dataset,
        metrics=RAGAS_METRICS,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=RunConfig(timeout=360),
    )
    eval_results[mode] = result.scores if hasattr(result, "scores") else result
    print(f"Results for {mode}:\n{eval_results[mode]}\n")

Evaluating baseline...


Evaluating:   0%|          | 0/35 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Results for baseline:
[{'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(1.0), 'answer_relevancy': np.float64(0.8670422740219837), 'context_entity_recall': 0.9999999900000002, 'context_precision': 0.9999999999, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(0.86), 'answer_relevancy': np.float64(0.6358112804880139), 'context_entity_recall': 0.0, 'context_precision': 0.8874999999778125, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(0.29), 'answer_relevancy': np.float64(0.338598900370033), 'context_entity_recall': 0.4999999975, 'context_precision': 0.49999999995, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(0.8), 'answer_relevancy': np.float64(0.38169984061389567), 'context_entity_recall': 0.0, 'context_

Evaluating:   0%|          | 0/35 [00:00<?, ?it/s]

...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Results for multi_query:
[{'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(1.0), 'answer_relevancy': np.float64(0.8670500365843045), 'context_entity_recall': 0.9999999900000002, 'context_precision': 0.9999999999, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(1.0), 'answer_relevancy': np.float64(0.5250422249373353), 'context_entity_recall': 0.0, 'context_precision': 0.6791666666496875, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(0.29), 'answer_relevancy': np.float64(0.3386122689323002), 'context_entity_recall': 0.4999999975, 'context_precision': 0.249999999975, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(0.8), 'answer_relevancy': np.float64(0.7975384926327842), 'context_entity_recall': 0.0, 'conte

Evaluating:   0%|          | 0/35 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Results for both:
[{'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(0.86), 'answer_relevancy': np.float64(0.7742635415106094), 'context_entity_recall': 0.9999999900000002, 'context_precision': 0.699999999965, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(1.0), 'answer_relevancy': np.float64(0.5250422249373353), 'context_entity_recall': 0.0, 'context_precision': 0.99999999998, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(0.33), 'answer_relevancy': np.float64(0.3922076552770051), 'context_entity_recall': 0.0, 'context_precision': 0.99999999995, 'noise_sensitivity(mode=relevant)': 0.0}, {'context_recall': 1.0, 'faithfulness': 1.0, 'factual_correctness(mode=f1)': np.float64(1.0), 'answer_relevancy': np.float64(0.7930481135952154), 'context_entity_recall': 0.0, 'context_precision': 0.99

## Comparison Summary

In [16]:
# Summary using np.nanmean (ignores NaN when averaging)
import numpy as np

summary_rows = []
for mode in MODES:
    scores = eval_results[mode]
    row = {"mode": mode}
    if isinstance(scores, list) and len(scores) > 0:
        for key in scores[0].keys():
            vals = [s[key] for s in scores if key in s and isinstance(s[key], (int, float, np.floating))]
            row[key] = float(np.nanmean(vals)) if vals else None
    else:
        row.update(scores or {})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

,mode,context_recall,faithfulness,factual_correctness(mode=f1),answer_relevancy,context_entity_recall,context_precision,noise_sensitivity(mode=relevant)
0,baseline,1.0,1.0,0.790,0.509249,0.3,0.817500,0.200000
1,rerank,1.0,1.0,0.800,0.574443,0.3,0.966667,0.266667
2,multi_query,1.0,1.0,0.818,0.570267,0.3,0.785833,0.200000
3,both,1.0,1.0,0.838,0.561531,0.2,0.906667,0.200000


In [ ]:
# Summary above shows mean scores per mode. Compare context_recall, context_precision, faithfulness.

## Summary Notes

| Metric | baseline | rerank | multi_query | both |
|--------|----------|--------|-------------|------|
| context_recall | 1.0 | 1.0 | 1.0 | 1.0 |
| faithfulness | 1.0 | 1.0 | 1.0 | 1.0 |
| factual_correctness | 0.79 | 0.80 | 0.82 | **0.84** |
| answer_relevancy | 0.51 | **0.57** | 0.57 | 0.56 |
| context_precision | 0.82 | **0.97** | 0.79 | 0.91 |
| context_entity_recall | 0.30 | 0.30 | 0.30 | 0.20 |
| noise_sensitivity (↓) | 0.20 | 0.27 | 0.20 | 0.20 |

### Improvements vs baseline

- **Rerank** – Largest gains: context_precision +0.15 (0.82 → 0.97), answer_relevancy +0.06. Best for precision and relevancy.
- **Both** – Best factual_correctness (0.84 vs 0.79). Strong context_precision (0.91). Best overall factual quality.
- **Multi_query** – Better factual_correctness (+0.03) but lower context_precision (−0.03). More diverse retrieval, less precise ranking.

### Recommendation

- **Rerank** – Best choice if you care most about context precision and answer relevancy.
- **Both** – Best choice if you care most about factual correctness and want a balanced setup.

## Notes

- **Reranking** requires `COHERE_API_KEY` in `.env`.
- **Multi-query** and **both** use the LLM for query expansion (same model as generation).
- Compare context_precision and context_recall across modes to see which improves retrieval quality.